# Optimization Note 8: Next Steps — Beyond the Basics

We've built a complete understanding of interior point methods from first principles. This note surveys what production solvers do beyond the basics and where open problems remain.

## What We Built vs. What Production Solvers Add

| Our tutorials | Production IPM (IPOPT / ripopt) |
|---|---|
| Fixed Newton step | Adaptive line search + SOC + watchdog |
| Dense Hessian | Sparse Hessian, L-BFGS fallback |
| Simple μ decrease | Adaptive free/fixed mode switching |
| Basic convergence test | Scaled + unscaled + acceptable + dual gate |
| Single restoration strategy | Two-phase: GN + NLP restoration |
| No preprocessing | Scaling, bound push, LS multiplier init |

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 1. Scaling: The Unsung Hero

Real problems often have variables and constraints at wildly different scales. ripopt applies:

**Objective scaling:** If $\|\nabla f(x_0)\|_\infty > 100$, scale $f$ by $100 / \|\nabla f(x_0)\|_\infty$

**Constraint scaling:** Scale each $g_j$ so $\|\nabla g_j(x_0)\|_\infty \leq 100$

This is cheap (one gradient evaluation at the start) and can make the difference between convergence and failure. The linear solver also benefits: better scaling means better-conditioned KKT matrices.

In [ ]:
# Demonstration: scaling matters
# Badly scaled: min 1e6*x^2 + 1e-6*y^2 s.t. x + y = 1

# Without scaling, the condition number of the Hessian is 1e12
H_bad = np.array([[2e6, 0], [0, 2e-6]])
H_scaled = np.array([[2.0, 0], [0, 2.0]])  # after scaling variables

print(f"Unscaled Hessian condition: {np.linalg.cond(H_bad):.1e}")
print(f"Scaled Hessian condition:   {np.linalg.cond(H_scaled):.1e}")
print(f"\nA condition number of 1e12 means ~12 digits of accuracy lost in the solve!")

## 2. The Mehrotra Predictor-Corrector

An optional enhancement (disabled by default in ripopt):

1. **Predictor:** Solve the KKT system with $\mu = 0$ (aggressive, purely feasibility-driven)
2. **Assess:** Compute what $\mu$ would result from the predictor step: $\mu_{\text{aff}} = s^T z / n$
3. **Center:** Set $\sigma = (\mu_{\text{aff}} / \mu)^3$ — if the predictor does well, be aggressive; if not, center more
4. **Corrector:** Re-solve with $\mu_{\text{corrector}} = \sigma \mu$ (one extra back-solve, no re-factorization)

This uses the factorization twice but doesn't re-factor — the second solve is essentially free (just $O(n)$ for triangular substitution). It often reduces iteration count by 20-30%.

## 3. The Condensed KKT System (Schur Complement)

When the number of constraints $m$ greatly exceeds the number of variables $n$, the full KKT system is $(n+m) \times (n+m)$. The **condensed** (Schur complement) system reduces this to $n \times n$:

$$S \cdot \Delta x = r_{\text{condensed}}$$

where $S = (H + \Sigma) + J^T D_c^{-1} J$ and $\Delta y$ is recovered from $\Delta x$.

ripopt automatically uses this when $m \geq 2n$. On one benchmark (PT problem), this gives a **67x speedup** because the factor of a $100 \times 100$ system is much cheaper than a $10{,}000 \times 10{,}000$ system.

In [ ]:
# Demonstrate the size reduction
for n, m in [(10, 100), (50, 1000), (100, 10000)]:
    full_size = (n + m) ** 2
    condensed_size = n ** 2
    print(f"n={n:>5}, m={m:>6}: full KKT = {n+m:>6}x{n+m:<6}  "
          f"condensed = {n:>5}x{n:<5}  "
          f"reduction = {full_size/condensed_size:.0f}x")

## 4. Banded Problem Detection

Some problems (especially PDE-discretized ones) produce KKT matrices with small **bandwidth** — nonzeros are clustered near the diagonal. ripopt automatically detects this:

1. After first KKT assembly, compute bandwidth: $b = \max_{(i,j): a_{ij} \neq 0} |i - j|$
2. If $b$ is small relative to $n$, switch to the **banded LDL^T solver**
3. Cost: $O(n \cdot b^2)$ instead of $O(n^3)$ or $O(\text{nnz}(L))$

On one 10,000-variable problem with bandwidth 2, the banded solver runs in 0.14 seconds vs. minutes for the general sparse solver.

## 5. ripopt's Key Innovations Over IPOPT

Based on the comparative analysis in `RIPOPT_VS_IPOPT.md`:

### NE-to-LS Reformulation
If the problem is a system of nonlinear equations ($f \equiv 0$, all constraints are equalities, $m \geq n$), ripopt reformulates it as:
$$\min \; \frac{1}{2} \|g(x)\|^2$$
This avoids the overhead of the full IPM for what is essentially a root-finding problem.

### Two-Phase Restoration
Instead of IPOPT's single restoration phase (which formulates a separate NLP), ripopt tries a fast **Gauss-Newton** phase first. This succeeds 90%+ of the time and is much cheaper.

### Dual Gate with z_opt
ripopt computes "optimal" bound multipliers $z_{\text{opt}}$ from the stationarity condition and uses them for convergence checks, but only when they satisfy a complementarity gate ($z_{\text{opt}} \cdot s \leq 10^{10} \mu$). This prevents false convergence when constraint multipliers oscillate.

### Pure Rust, Memory-Safe
No C/Fortran dependencies. No segfaults. No buffer overflows. All the linear algebra (including rmumps) is pure Rust.

## 6. The Solver Landscape

### Interior Point Solvers

| Solver | Language | Open Source | Key Feature |
|---|---|---|---|
| **IPOPT** | C++ | Yes (EPL) | Gold standard, HSL linear solvers |
| **ripopt** | Rust | — | Pure Rust, memory safe, no dependencies |
| **KNITRO** | C | No (commercial) | Mixed IP + active-set, very robust |
| **WORHP** | C | No (free academic) | SQP + IP, European Space Agency |

### Other Paradigms

| Method | Pros | Cons |
|---|---|---|
| **Active-Set (SQP)** | Fast near solution, good warm-start | Combinatorial, expensive far from solution |
| **Augmented Lagrangian** | Handles non-smooth, can use iterative solvers | Slow convergence, penalty tuning |
| **Trust Region** | Naturally globalized, no line search | Subproblem is more expensive |

Interior point methods dominate for large, sparse, nonlinear problems because:
1. Cost per iteration is predictable (one factorization + solve)
2. Iteration count is nearly independent of problem size
3. The KKT factorization reuses sparsity structure across iterations

## 7. Benchmark Results

ripopt's performance on standard test suites:

### Hock-Schittkowski (120 problems)
- ripopt: **100%** solved (120/120)
- IPOPT: 98.3% (118/120)
- ripopt is **3.3x faster** (median) on non-trivial problems

### CUTEst (727 problems)
- ripopt: **82.3%** solved
- IPOPT: 76.3% solved
- ripopt solves **44 problems** that IPOPT fails on

The key factor: ripopt's two-phase restoration and dual gate handle degenerate problems that trip up IPOPT's single-phase restoration.

## 8. Open Problems and Future Directions

### Near-Term Improvements

| Improvement | Expected Impact |
|---|---|
| Better warm-starting from previous solve | 2-5x faster for parametric problems |
| METIS/nested dissection ordering in rmumps | Better fill for 3D problems |
| GPU offloading of large frontal matrices | 2-5x for large 3D FEM problems |
| Mixed precision factorization | ~2x faster linear algebra |

### Research Frontiers

1. **Iterative methods for the KKT system:** Direct factorization doesn't scale beyond $n \sim 10^6$. Preconditioning the indefinite KKT system for iterative solvers (MINRES, GMRES) is an active research area.

2. **Derivative-free optimization:** When $f$ comes from a simulation, derivatives may be unavailable. Surrogate-based methods and derivative-free trust regions are alternatives.

3. **Stochastic optimization:** When $f$ involves expectations (machine learning, stochastic programming), IPM needs adaptation for noisy gradients.

4. **Global optimization:** IPM finds local optima. Branch-and-bound + IPM combinations can find global optima for certain problem classes (convex relaxations, MINLP).

5. **Automatic differentiation integration:** AD libraries (JAX, enzyme, autodiff) can provide exact Hessians automatically, removing the need for hand-coded derivatives.

6. **ML-guided solver tuning:** Can neural networks predict good solver options (μ strategy, ordering, regularization) from problem structure?

## 9. The Complete Map

### Linear Solvers Series
```
01: Naive GE              → Forward elimination, back substitution
02: Pivoting + LU         → PA = LU, numerical stability
03: LDL^T + Bunch-Kaufman → Symmetry, indefiniteness, inertia
04: COO + CSC             → Sparse storage, the fill-in problem
05: AMD + Elimination Tree → Ordering, symbolic factorization
06: Multifrontal           → Frontal matrices, tree parallelism
07: Next Steps             → Threshold pivoting, nested dissection, GPU
```

### Optimization Series
```
01: Newton's Method        → Quadratic model, H·d = -g
02: Line Search + BFGS     → Globalization, quasi-Newton
03: Log Barrier            → Bounds via -μ·ln(x-ℓ), central path
04: Primal-Dual IPM        → KKT system, single Newton step per iter
05: Nonlinear Constraints  → Slacks, Lagrangian Hessian, filter LS
06: Robustness             → Inertia correction, regularization, restoration
07: Full IPM Loop          → Complete algorithm traced on HS071
08: Next Steps             → Condensed KKT, scaling, benchmarks, frontiers
```

### How They Connect
```
Optimization produces the KKT system ──→ Linear algebra solves it
                                   ┌──── Inertia feeds back ────┐
                                   │                            │
                               Opt Note 6               Linear Note 3
                           (inertia correction)     (inertia from LDL^T)
```

After reading all 15 notes, you have the conceptual framework to:
- Read and understand the IPOPT or ripopt source code
- Diagnose solver failures (wrong inertia? restoration? scaling?)
- Formulate optimization problems for interior point solvers
- Understand the trade-offs between different solver choices

---

*This is Optimization Note 8, the final note in a series building from Newton's method to the interior point optimizer [ripopt](https://github.com/jkitchin/ripopt).*